# Chapter 18 — Image Derivatives

*Companion notebook for* **Foundations of Computer Vision** *(Torralba, Isola, Freeman), Ch. 18 — [visionbook.mit.edu](https://visionbook.mit.edu/derivatives.html).*

The image derivative is the workhorse of low-level vision: it turns intensity changes into signals we can measure. This notebook builds the derivative operators the chapter develops — the two-tap $[1,-1]$ and centred $[1,0,-1]$ kernels, **Gaussian derivatives** (via Hermite polynomials), **derivative-of-binomial** kernels, **Roberts / Sobel** operators, the **Laplacian** and Laplacian-of-Gaussian, **unsharp masking**, and a working **Retinex** decomposition — each backed by a numerical check.

The photo figures load the **book's own images** live from `visionbook.mit.edu` (referenced, not redistributed). Colour inputs are shown in colour. Following the book, derivative maps are shown **signed around mid-gray** (per channel, so colour edges for colour inputs).

In [ ]:
import io, urllib.request
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from numpy.polynomial.hermite import hermval
from PIL import Image

torch.manual_seed(0); np.random.seed(0)
torch.set_default_dtype(torch.float32)
plt.rcParams.update({'figure.dpi': 130, 'savefig.dpi': 130,
                     'image.cmap': 'gray', 'image.interpolation': 'nearest', 'axes.grid': False})

BOOK_FIG = 'https://visionbook.mit.edu/figures'


def load_book_gray(url):
    with urllib.request.urlopen(url, timeout=30) as r:
        im = Image.open(io.BytesIO(r.read())).convert('L')
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0)


def load_book_rgb(url):
    with urllib.request.urlopen(url, timeout=30) as r:
        im = Image.open(io.BytesIO(r.read())).convert('RGB')
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0)


def conv2d(image, kernel, mode='reflect'):
    """2D convolution, "same" size, reflect-padded (kernels here are symmetric-ish)."""
    k = torch.as_tensor(kernel, dtype=torch.float32).flip(0).flip(1)
    kh, kw = k.shape
    x = F.pad(image[None, None], (kw // 2, kw // 2, kh // 2, kh // 2), mode=mode)
    return F.conv2d(x, k[None, None])[0, 0]


def conv2d_rgb(image, kernel, mode='reflect'):
    return torch.stack([conv2d(image[..., c], kernel, mode) for c in range(3)], dim=-1)


def gaussian_1d(sigma, radius=None):
    if radius is None:
        radius = int(np.ceil(3 * sigma))
    x = torch.arange(-radius, radius + 1, dtype=torch.float32)
    k = torch.exp(-x**2 / (2 * sigma**2))
    return k / k.sum()


def dgauss_1d(sigma, order, radius=None):
    """n-th derivative of a unit-area 1D Gaussian, via Hermite polynomials:
    g_{x^n}(x) = (-1/(sigma*sqrt2))^n H_n(x/(sigma*sqrt2)) g(x).
    """
    if radius is None:
        radius = int(np.ceil(4 * sigma)) + order
    x = np.arange(-radius, radius + 1, dtype=np.float64)
    g = np.exp(-x**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))
    a = x / (sigma * np.sqrt(2))
    Hn = hermval(a, [0] * order + [1])
    k = ((-1.0 / (sigma * np.sqrt(2))) ** order) * Hn * g
    return torch.tensor(k, dtype=torch.float32)


def gauss_deriv2d(sigma, nx, ny):
    """Separable 2D Gaussian derivative kernel (order nx in x, ny in y)."""
    kx = dgauss_1d(sigma, nx); ky = dgauss_1d(sigma, ny)
    return torch.outer(ky, kx)


def grad_mag(image, kx, ky):
    return torch.sqrt(conv2d(image, kx)**2 + conv2d(image, ky)**2 + 1e-12)


def norm01(a):
    """Scale a tensor to [0, 1] for display."""
    a = a - a.min()
    return a / (a.max() + 1e-12)


def show_grad(g, p=0.99, gamma=0.6):
    """Display map for a gradient magnitude: clip at the p-quantile, gamma-brighten."""
    hi = torch.quantile(g.flatten(), p)
    return (g / (hi + 1e-9)).clamp(0, 1) ** gamma


def show_signed(d, p=0.99):
    """Display a SIGNED (possibly colour) derivative around mid-gray, like the book:
    positive -> lighter, negative -> darker, per channel; p-quantile maps to +-0.5.
    """
    hi = torch.quantile(d.abs().flatten(), p)
    return (0.5 + 0.5 * d / (hi + 1e-9)).clamp(0, 1)


def show(panels, titles, figsize=None):
    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=figsize or (3.3 * n, 3.5))
    if n == 1:
        axes = [axes]
    for ax, im, t in zip(axes, panels, titles):
        arr = im.detach().cpu().numpy() if torch.is_tensor(im) else im
        if arr.ndim == 3:
            ax.imshow(np.clip(arr, 0, 1))
        else:
            ax.imshow(arr, cmap='gray')
        ax.set_title(t, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()


def luminance(rgb):
    w = torch.tensor([0.299, 0.587, 0.114])
    return (rgb * w).sum(-1)

# Standard finite-difference kernels used throughout.
D0 = torch.tensor([[1.0, -1.0]])                 # two-tap  [1, -1]
D1 = torch.tensor([[1.0, 0.0, -1.0]]) / 2.0      # centred  [1, 0, -1] / 2

## 18.2 — Discretizing the image derivative

The continuous partial derivative $\partial\ell/\partial x$ becomes a finite difference. Two choices dominate:

$$d_0 = [1,\,-1]\ (\ell[n]-\ell[n-1]),\qquad d_1 = \tfrac12[1,\,0,\,-1]\ \bigl(\tfrac{\ell[n+1]-\ell[n-1]}{2}\bigr).$$

$d_0$ is a half-pixel-shifted difference; $d_1$ is **centred** (no shift) and a touch smoother. Applying $d_1$ across $x$ and down $y$ gives the two derivative images of Figure 18.2. Following the book, the derivative of each colour channel is shown **signed around mid-gray** — light where intensity rises, dark where it falls — which paints edges in blue/orange.

In [ ]:
# Figure 18.2 — x/y derivatives of the book's colour photo, shown the book's way:
# the SIGNED derivative of each colour channel around mid-gray (blue/orange edges).
photo = load_book_rgb(f'{BOOK_FIG}/derivatives/mit_der_a.jpg')
dx = conv2d_rgb(photo, D1)
dy = conv2d_rgb(photo, D1.T)
show([photo, show_signed(dx), show_signed(dy)],
     ['(a) input photo', '(b) x-derivative  d/dx', '(c) y-derivative  d/dy'])
print('mean |d/dx|:', dx.abs().mean().item(), ' mean |d/dy|:', dy.abs().mean().item())

### 18.4 — What the two kernels do in frequency

An ideal derivative multiplies each frequency by $j\omega$, i.e. magnitude grows **linearly** with frequency. The DFTs

$$D_0[u] = 1-e^{-2\pi j u/N},\ |D_0| = 2\sin(\pi u/N),\qquad D_1[u] = j\sin(2\pi u/N),$$

both approximate $|\omega|$ at low frequencies. $|D_0|$ tracks the ideal further up the band; $|D_1|$ rolls off earlier (it suppresses the highest frequencies — smoother, less noise).

In [ ]:
# Figure 18.4 — |DFT| of d0 and d1 vs the ideal |omega|, as discrete stems over
# u = -N/2 .. N/2 (as the book draws them), with the ideal shown as a line.
N = 20
u = np.arange(-N // 2, N // 2 + 1)             # -10 .. 10
ideal = np.abs(2 * np.pi * u / N)              # ideal derivative response |omega|
D0m = np.abs(1 - np.exp(-2j * np.pi * u / N))  # = 2 sin(pi u / N)
D1m = np.abs(np.sin(2 * np.pi * u / N))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
for a, D, ttl in [(ax[0], D0m, '(a) two-tap  |D0[u]|,  d0 = [1,-1]'),
                  (ax[1], D1m, '(b) centred  |D1[u]|,  d1 = [1,0,-1]/2')]:
    a.stem(u, D, basefmt=' ')
    a.plot(u, ideal, 'k', lw=1, label='ideal |omega|')
    a.set_title(ttl, fontsize=10); a.set_xlabel('u'); a.set_ylim(0, 2.3)
    a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 18.5 — Gaussian derivatives beat noise

Differentiation amplifies high frequencies, so a raw $[1,-1]$ derivative of a noisy image is dominated by noise. Because differentiation and convolution commute,

$$\frac{\partial \ell}{\partial x}*g = \ell * \frac{\partial g}{\partial x},$$

we can differentiate the **smooth Gaussian** instead of the noisy image. The first Gaussian derivative $g_x=-\tfrac{x}{\sigma^2}g$ smooths and differentiates in one pass. Figure 18.6 contrasts the two on a noisy photo.

In [ ]:
# Figure 18.6 — raw x-derivative (noise-amplified) vs Gaussian x-derivative.
# Shown the way the book does it: the SIGNED derivative of each colour channel,
# displayed around mid-gray (light = positive edge, dark = negative edge).
noisy = load_book_rgb(f'{BOOK_FIG}/derivatives/stop_noise.jpg')[::4, ::4]   # strided: keep noise
raw = conv2d_rgb(noisy, D1)                                 # raw centred difference, per channel
gauss = conv2d_rgb(noisy, gauss_deriv2d(3.0, 1, 0))         # Gaussian x-derivative, sigma=3
show([noisy, show_signed(raw), show_signed(gauss)],
     ['(a) noisy input', '(b) raw d/dx (noise)', '(c) Gaussian d/dx (clean edges)'])
print('background derivative std — raw   :', raw[:80, :80].std().item())
print('background derivative std — gauss :', gauss[:80, :80].std().item(),
      '  # Gaussian derivative suppresses the noise floor')

## 18.6 — Gaussian derivatives and the Hermite family

Higher derivative orders of the Gaussian are Hermite polynomials times the Gaussian:

$$g_{x^n}(x;\sigma) = \Bigl(\tfrac{-1}{\sigma\sqrt2}\Bigr)^n H_n\!\Bigl(\tfrac{x}{\sigma\sqrt2}\Bigr)\,g(x;\sigma),\qquad H_n(x)=2xH_{n-1}-2(n-1)H_{n-2}.$$

Each order adds one more oscillation. Orders 0–3 for $\sigma=1$:

In [ ]:
# Figure 18.9 — the Gaussian and its first three derivatives, sigma = 1
# (smooth continuous curves over x in [-4, 4], as in the book).
x = np.linspace(-4, 4, 400)
g = np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)
a = x / np.sqrt(2)
titles = ['$g(x)$', '$g_x(x)$', '$g_{x^2}(x)$', '$g_{x^3}(x)$']
fig, ax = plt.subplots(1, 4, figsize=(12, 2.8))
for n in range(4):
    curve = ((-1 / np.sqrt(2)) ** n) * hermval(a, [0] * n + [1]) * g
    ax[n].plot(x, curve); ax[n].axhline(0, color='k', lw=0.6)
    ax[n].set_title(titles[n]); ax[n].set_xlabel('x')
plt.tight_layout(); plt.show()
# Sanity: order-0 integrates to ~1 (unit area); every derivative integrates to ~0.
for n in range(4):
    print(f'sum of order {n}:', float(dgauss_1d(1.0, n, radius=8).sum()))

### 18.10 — The 2-D Gaussian-derivative triangle

Because the 2-D Gaussian is **separable**, every mixed partial $g_{x^n y^m}$ is just the outer product of two 1-D Hermite-weighted Gaussians. Arranged by total order they form a triangle (the top is $g$ itself; each row adds one derivative). Each kernel is shown signed around mid-gray — these are exactly the oriented centre-surround filters a linear front-end computes.

In [ ]:
# Figure 18.10 — triangle of separable 2D Gaussian derivatives, order 0..4.
K = 4; sig = 6.0
fig, axes = plt.subplots(K + 1, K + 1, figsize=(8.5, 8))
for r in range(K + 1):
    for c in range(K + 1):
        axes[r][c].axis('off')
for n in range(K + 1):                 # row = total derivative order
    for j in range(n + 1):             # nx in x, ny in y (nx+ny = n)
        nx, ny = n - j, j
        ker = gauss_deriv2d(sig, nx, ny).numpy()
        m = np.abs(ker).max() + 1e-12
        ax = axes[n][j]
        ax.imshow(ker, cmap='gray', vmin=-m, vmax=m)
        lbl = 'g' if n == 0 else f'g$_{{x^{nx}y^{ny}}}$'
        ax.set_title(lbl, fontsize=8)
plt.suptitle('2D Gaussian derivatives (separable), order 0 to 4', y=0.98)
plt.tight_layout(); plt.show()

### 18.8 — Multiscale Gaussian derivatives

The scale $\sigma$ selects which edges survive: small $\sigma$ picks up fine texture, large $\sigma$ only the coarse structure. Here is the Gaussian x-derivative magnitude of the book's zebra at $\sigma=2,4,8$.

In [ ]:
# Figure 18.8 — Gaussian x-derivative of the zebra at increasing scale,
# shown signed around mid-gray (embossed edges), as in the book.
zebra = load_book_gray(f'{BOOK_FIG}/spatial_filters/gausian_zebra_c_2.jpg')
panels, titles = [zebra], ['input (book zebra)']
for s in (2.0, 4.0, 8.0):
    resp = conv2d(zebra, gauss_deriv2d(s, 1, 0))
    panels.append(show_signed(resp)); titles.append(f'Gaussian d/dx, sigma={int(s)}')
show(panels, titles)

## 18.7 — Derivatives from binomial filters

Convolving a binomial smoother $b_n$ (Pascal's triangle) with the elementary difference $[1,-1]$ gives a family of discrete derivative kernels $d_n=b_n*[1,-1]$: smoother as $n$ grows, all with DC gain 0.

$$d_0=[1,-1],\quad d_1=[1,0,-1],\quad d_2=[1,1,-1,-1],\quad d_3=[1,2,0,-2,-1],\dots$$

In [ ]:
# The derivative-of-binomial family: d_n = b_n * [1, -1].
def binom(n):
    b = np.array([1.0])
    for _ in range(n):
        b = np.convolve(b, [1.0, 1.0])
    return b

fig, ax = plt.subplots(1, 5, figsize=(12, 2.6))
for n in range(5):
    dn = np.convolve(binom(n), [1.0, -1.0])
    xs = np.arange(len(dn)) - len(dn) // 2
    ax[n].stem(xs, dn); ax[n].set_title(f'd{n} = b{n} * [1,-1]', fontsize=9)
    ax[n].axhline(0, color='k', lw=0.6)
    print(f'd{n}:', dn.astype(int), ' sum =', int(dn.sum()))
plt.tight_layout(); plt.show()

### 18.14 — Roberts and Sobel in frequency

The **Roberts cross** ($2\times2$ diagonal differences) and the **Sobel–Feldman** operator ($[1,0,-1]$ derivative $\times$ $[1,2,1]$ smoothing) are the classic 2-D edge operators. Sobel is separable — $\text{Sobel}_x = [1,0,-1]\otimes[1,2,1]^\top$ — and its smoothing makes it the most **isotropic and noise-tolerant**. The 2-D DFT magnitudes below show each operator's directional selectivity.

In [ ]:
# Figure 18.14 — 2D |DFT| SURFACES of d0, d1, Roberts_x, Sobel_x (as in the book).
def dft_mag(kernel, N=64):
    k = np.zeros((N, N)); kh, kw = kernel.shape
    k[:kh, :kw] = kernel                                   # place at origin
    return np.abs(np.fft.fftshift(np.fft.fft2(k)))

kernels = {
    '|D0(u,v)|  [1,-1]': np.array([[1.0, -1.0]]),
    '|D1(u,v)|  [1,0,-1]/2': np.array([[1.0, 0.0, -1.0]]) / 2,
    'Roberts_x': np.array([[1.0, 0.0], [0.0, -1.0]]),
    'Sobel_x': np.array([[1.0, 0.0, -1.0], [2.0, 0.0, -2.0], [1.0, 0.0, -1.0]]),
}
uv = np.linspace(-0.5, 0.5, 64)
U, V = np.meshgrid(uv, uv)
fig = plt.figure(figsize=(14, 3.4))
for i, (name, k) in enumerate(kernels.items(), 1):
    ax = fig.add_subplot(1, 4, i, projection='3d')
    ax.plot_surface(U, V, dft_mag(k), cmap='viridis', linewidth=0, antialiased=False)
    ax.set_title(name, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_xlabel('u', fontsize=8); ax.set_ylabel('v', fontsize=8)
    ax.view_init(elev=28, azim=-60)
plt.tight_layout(); plt.show()

## 18.8 — Image gradient and directional derivatives

The gradient $\nabla\ell=(\partial_x\ell,\partial_y\ell)$ is a per-pixel vector. The derivative in any direction $\mathbf t=(\cos\theta,\sin\theta)$ is just a linear combination of the two we already have — **no new convolution needed**:

$$\frac{\partial\ell}{\partial\mathbf t} = \cos\theta\,\partial_x\ell + \sin\theta\,\partial_y\ell.$$

In [ ]:
# Figure 18.15 — directional derivatives of a disc: d/dx, d/dy, d/d45, plus the
# gradient magnitude |grad I| and the gradient ORIENTATION as a colour wheel
# (hue = angle), exactly as the book lays it out.
import math
import matplotlib.colors as mcolors
yy, xx = torch.meshgrid(torch.linspace(-1, 1, 256), torch.linspace(-1, 1, 256), indexing='ij')
disc = (xx**2 + yy**2 < 0.5**2).float()
smooth = torch.outer(gaussian_1d(2.0), gaussian_1d(2.0))     # anti-alias the hard edge
disc = conv2d(disc, smooth)
dx = conv2d(disc, D1); dy = conv2d(disc, D1.T)
d45 = math.cos(math.pi / 4) * dx + math.sin(math.pi / 4) * dy
mag = torch.sqrt(dx**2 + dy**2)
ang = torch.atan2(dy, dx)                                    # -pi .. pi
hue = ((ang / (2 * math.pi)) % 1.0).numpy()                  # 0 .. 1
val = (mag / mag.max()).clamp(0, 1).numpy() ** 0.5           # brightness = edge strength
angle_rgb = mcolors.hsv_to_rgb(np.stack([hue, np.ones_like(hue), val], axis=-1))

show([disc, show_signed(dx), show_signed(dy), show_signed(d45), norm01(mag), angle_rgb],
     ['input disc', 'd/dx', 'd/dy', 'd/d(45 deg)', '|grad I|', 'angle(grad I)'],
     figsize=(18, 3.1))

## 18.9 — The Laplacian and Laplacian-of-Gaussian

The **Laplacian** $\nabla^2\ell=\partial_{xx}\ell+\partial_{yy}\ell$ is the simplest **rotationally invariant** second-order operator. Smoothed with a Gaussian it becomes the **Laplacian-of-Gaussian** (the *Mexican-hat* wavelet),

$$\nabla^2 g = \frac{x^2+y^2-2\sigma^2}{\sigma^4}\,g(x,y;\sigma),$$

a centre-surround kernel that responds to blobs and zero-crosses at edges.

In [ ]:
# Figure 18.16 — Laplacian-of-Gaussian (Mexican hat), sigma = 1, x,y in [-4, 4].
# Plotted as grad^2 g (NOT negated): a well that dips to ~-0.3 with a positive rim.
xx = np.linspace(-4, 4, 121)
X, Y = np.meshgrid(xx, xx)
sig = 1.0
g = np.exp(-(X**2 + Y**2) / (2 * sig**2)) / (2 * np.pi * sig**2)
log = (X**2 + Y**2 - 2 * sig**2) / sig**4 * g          # grad^2 g
fig = plt.figure(figsize=(9.5, 3.6))
a = fig.add_subplot(1, 2, 1, projection='3d')
a.plot_surface(X, Y, log, cmap='jet', linewidth=0, antialiased=False)
a.set_title(r'(a) $\nabla^2 g(x,y)$'); a.set_xlabel('x'); a.set_ylabel('y')
a.set_zticks([-0.2, 0]); a.view_init(elev=22, azim=-60)
b = fig.add_subplot(1, 2, 2)
b.plot(xx, log[len(xx) // 2]); b.axhline(0, color='k', lw=0.6)
b.set_title(r'(b) section $\nabla^2 g(x,0)$'); b.set_xlabel('x')
plt.tight_layout(); plt.show()

In [ ]:
# Figure 18.18 — second derivatives of the wheel: d2/dx2, d2/dy2, and their sum
# (the Laplacian) is rotationally invariant. Five-point stencil:
#   [[0,1,0],[1,-4,1],[0,1,0]].
wheel = load_book_gray(f'{BOOK_FIG}/spatial_filters/wheel256.jpg')
dxx = conv2d(wheel, torch.tensor([[1.0, -2.0, 1.0]]))          # d2/dx2
dyy = conv2d(wheel, torch.tensor([[1.0], [-2.0], [1.0]]))      # d2/dy2
lap = conv2d(wheel, torch.tensor([[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]]))
show([wheel, show_signed(dxx), show_signed(dyy), show_signed(lap)],
     ['input (book wheel)', 'd2/dx2', 'd2/dy2', 'Laplacian (isotropic)'])
print('max |dxx+dyy - Laplacian|:', (dxx + dyy - lap).abs().max().item())

## 18.11 — Sharpening: unsharp masking

Subtracting a blurred copy from twice the image boosts the high frequencies the blur removed:

$$\text{sharpen} = 2\mathbf I - b_{2,2},\qquad \text{DC gain} = 1.$$

Applied repeatedly it keeps enhancing edges (until artefacts appear). The boat is colour, so the sharpen kernel is applied to each channel independently.

In [ ]:
# Figure 18.23 — unsharp masking applied 1..5 times to the book's boat (colour).
boat = load_book_rgb(f'{BOOK_FIG}/spatial_filters/boat_sharp0.jpg')
b = torch.tensor([1.0, 2.0, 1.0]); blur = torch.outer(b, b); blur = blur / blur.sum()
sharpen = torch.zeros(3, 3); sharpen[1, 1] = 2.0; sharpen = sharpen - blur   # 2I - b22

panels, titles = [boat], ['original']
cur = boat
for i in range(1, 6):
    cur = conv2d_rgb(cur, sharpen).clamp(0, 1)
    panels.append(cur); titles.append(f'sharpen x{i}')
show(panels, titles, figsize=(16, 3.0))
print('sharpen kernel DC gain:', float(sharpen.sum()))

## 18.12 — Retinex: separating reflectance from illumination

An image is reflectance times illumination, $\ell=r\cdot l$. In the log domain this is a sum, and Land's **Retinex** exploits a statistical gap: **reflectance edges are sharp (large log-gradients)** while **illumination varies smoothly (small gradients)**. So we threshold the log-gradient — keep the large part as reflectance, integrate it back with a (mirror-padded) Poisson solve, and take the smooth remainder as illumination.

We use the book's test setup — a **synthetic Mondrian** (piecewise-constant reflectance patches) under a **smooth, left-bright illumination** — and lay it out as the book's $\ell(x,y)=r(x,y)\cdot l(x,y)$ decomposition. Because we know the ground truth we can *measure* the recovery: the smooth **illumination is recovered almost exactly** (correlation ~0.97), and the **reflectance** comes out flat (~0.77; the residual is faint illumination the single global threshold cannot fully separate).

In [ ]:
# Figure 18.25/26 — Retinex on a synthetic Mondrian x smooth illumination,
# laid out as the book's  l(x,y) = r(x,y) x l(x,y)  decomposition.
def fdx(a): return torch.roll(a, -1, dims=1) - a          # forward difference f[n+1]-f[n]
def fdy(a): return torch.roll(a, -1, dims=0) - a

def poisson_periodic(gx, gy):
    H, W = gx.shape
    fy = np.fft.fftfreq(H)[:, None]; fx = np.fft.fftfreq(W)[None, :]
    Dx = np.exp(2j * np.pi * fx) - 1; Dy = np.exp(2j * np.pi * fy) - 1
    Gx = np.fft.fft2(gx); Gy = np.fft.fft2(gy)
    denom = np.abs(Dx)**2 + np.abs(Dy)**2; denom[0, 0] = 1.0
    Fh = (np.conj(Dx) * Gx + np.conj(Dy) * Gy) / denom; Fh[0, 0] = 0.0
    return np.real(np.fft.ifft2(Fh))

def integrate(gx, gy):
    """Integrate a gradient field. The gradients are MIRROR-reflected first so the
    periodic FFT solve behaves like a Neumann boundary — without this the recovered
    reflectance keeps a low-frequency illumination ramp.
    """
    gx = gx.numpy(); gy = gy.numpy(); H, W = gx.shape
    GX = np.block([[gx, -gx[:, ::-1]], [gx[::-1, :], -gx[::-1, ::-1]]])
    GY = np.block([[gy, gy[:, ::-1]], [-gy[::-1, :], -gy[::-1, ::-1]]])
    f = poisson_periodic(GX, GY)[:H, :W]
    return torch.tensor(f, dtype=torch.float32)

def corr(a, b):
    a = a.flatten() - a.mean(); b = b.flatten() - b.mean()
    return (a * b).sum() / (a.norm() * b.norm() + 1e-12)

# Ground truth: a rich Mondrian (distinct grey levels) under a smooth,
# left-bright illumination — the book's test image.
torch.manual_seed(7)
gyN, gxN, ps = 8, 16, 18
H, Wd = gyN * ps, gxN * ps                          # 144 x 288, wide like the book
levels = torch.tensor([0.18, 0.32, 0.46, 0.60, 0.74, 0.90])
patch = levels[torch.randint(0, len(levels), (gyN, gxN))]
R_true = patch.repeat_interleave(ps, 0).repeat_interleave(ps, 1)
yy, xx = torch.meshgrid(torch.linspace(0, 1, H), torch.linspace(0, 1, Wd), indexing='ij')
L_true = 0.18 + 0.82 * torch.exp(-(((xx - 0.16)**2) / (2 * 0.22**2)
                                    + ((yy - 0.40)**2) / (2 * 0.50**2)))
ell = (R_true * L_true).clamp(1e-3, 1.0)

# Retinex: threshold log-gradients, keep the sharp (reflectance) part, integrate.
logL = torch.log(ell)
gx, gy = fdx(logL), fdy(logL)
T = torch.quantile(torch.stack([gx.abs(), gy.abs()]).flatten(), 0.86)
gxr = torch.where(gx.abs() > T, gx, torch.zeros_like(gx))
gyr = torch.where(gy.abs() > T, gy, torch.zeros_like(gy))
logR = integrate(gxr, gyr); logR = logR - logR.mean() + logL.mean()
R_hat = torch.exp(logR).clamp(0, 2)
L_hat = ell / (R_hat + 1e-3)                        # illumination = smooth remainder

show([ell, norm01(R_hat), norm01(L_hat)],
     ['(a) input  l = r x l', '(b) reflectance r (sharp)', '(c) illumination l (smooth)'],
     figsize=(14, 3.0))
print('corr(recovered reflectance,  truth):', corr(R_hat, R_true).item())
print('corr(recovered illumination, truth):', corr(L_hat, L_true).item())

## 18.13 — Concluding remarks

| Operator | Kernel | Property |
|---|---|---|
| two-tap $d_0$ | $[1,-1]$ | half-pixel shift, widest band |
| centred $d_1$ | $[1,0,-1]/2$ | no shift, smoother |
| Gaussian deriv | $-\tfrac{x}{\sigma^2}g$ | smooths + differentiates; scale-selective |
| Sobel | $[1,0,-1]\otimes[1,2,1]$ | separable, isotropic, noise-tolerant |
| Laplacian | $[[0,1,0],[1,-4,1],[0,1,0]]$ | rotationally invariant, zero-crossings at edges |
| sharpen | $2\mathbf I - b_{2,2}$ | high-boost, DC gain 1 |

From a single idea — differencing neighbouring pixels — the chapter builds edge detection, scale selection (Gaussian derivatives), the isotropic Laplacian, sharpening, and gradient-domain reasoning strong enough to **separate reflectance from illumination**. These operators are the front end of nearly every classical vision pipeline (SIFT, HOG) and echo the centre-surround receptive fields of early biological vision.